In [82]:
import pandas as pd

from statsmodels.tsa.stattools import adfuller

In [83]:
df = pd.read_csv(
    '../data/raw/household_power_consumption.txt',
    sep=';',
    na_values=['?'],
    low_memory=False
)

In [84]:
df['datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%Y %H:%M:%S'
)

df.set_index('datetime', inplace=True)
df.sort_index(inplace=True)

df.drop(['Date', 'Time'], axis=1, inplace=True)

In [85]:
df = df.apply(pd.to_numeric)

In [86]:
df.isnull().sum()

Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
dtype: int64

In [87]:
df.interpolate(method='time', inplace=True)

,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0
...,...,...,...,...,...,...,...
2010-11-26 20:58:00,0.946,0.000,240.43,4.0,0.0,0.0,0.0
2010-11-26 20:59:00,0.944,0.000,240.00,4.0,0.0,0.0,0.0
2010-11-26 21:00:00,0.938,0.000,239.82,3.8,0.0,0.0,0.0


In [88]:
df.isnull().sum()

Global_active_power      0
Global_reactive_power    0
Voltage                  0
Global_intensity         0
Sub_metering_1           0
Sub_metering_2           0
Sub_metering_3           0
dtype: int64

In [89]:
energy = df['Global_active_power']

energy_hourly = energy.resample('h').mean()

energy_hourly.head()

datetime
2006-12-16 17:00:00    4.222889
2006-12-16 18:00:00    3.632200
2006-12-16 19:00:00    3.400233
2006-12-16 20:00:00    3.268567
2006-12-16 21:00:00    3.056467
Freq: h, Name: Global_active_power, dtype: float64

In [90]:
energy_hourly.isnull().sum()

np.int64(0)

### Stationarity Test (ADF Test)

The Augmented Dickey-Fuller (ADF) test is used to determine whether the time series is stationary.

- Null Hypothesis: The series is non-stationary
- Alternative Hypothesis: The series is stationary

In [91]:
result = adfuller(energy_hourly)

print('ADF Statistic:', result[0])
print('p-value:', result[1])

ADF Statistic: -14.371344689221477
p-value: 9.482687472633596e-27


### Interpretation

The p-value is significantly less than 0.05, which indicates that the null hypothesis can be rejected. Therefore, the time series is stationary.

This suggests that differencing may not be strictly required for this dataset.

In [92]:
energy_diff = energy_hourly.diff().dropna()

In [93]:
result = adfuller(energy_diff)

print('ADF Statistic:', result[0])
print('p-value:', result[1])

ADF Statistic: -42.81070644158096
p-value: 0.0


### Stationarity Analysis

The Augmented Dickey-Fuller (ADF) test was used to assess stationarity.

- The original time series was non-stationary (p-value > 0.05)
- After applying first-order differencing, the series became stationary (p-value < 0.05)

This confirms the need for differencing when using models like SARIMA.